In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.arima.model import ARIMA
from math import sqrt
import warnings
import pyodbc

In [ ]:
# Tải Dữ Liệu Từ Database
conn = pyodbc.connect("Driver={SQL Server};Server=SUBIN;Database=SAS;Trusted_Connection=yes;")
cursor = conn.cursor()

query = "SELECT sale_date, revenue FROM Sales ORDER BY sale_date"
df = pd.read_sql(query, conn)
df['sale_date'] = pd.to_datetime(df['sale_date'])

monthly_data = df.resample('ME', on='sale_date').sum().reset_index()

In [ ]:
# Mô Hình Linear Regression và Random Forest
warnings.filterwarnings('ignore')

# Tiền Xử Lý: Thêm month_index
monthly_data['month_index'] = np.arange(len(monthly_data))
X = monthly_data[['month_index']]
y = monthly_data['revenue']

# Chia Dữ Liệu Train/Test (80/20) Cho Đánh Giá
train_size = int(len(y) * 0.8)
y_train, y_test = y[:train_size], y[train_size:]
X_train, X_test = X[:train_size], X[train_size:]

# Đánh Giá Linear Regression
lr_model_eval = LinearRegression()
lr_model_eval.fit(X_train.values, y_train)
lr_pred_eval = lr_model_eval.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred_eval)
lr_rmse = sqrt(mean_squared_error(y_test, lr_pred_eval))
print(f"Linear Regression Evaluation - MAE: {lr_mae:.2f}, RMSE: {lr_rmse:.2f}")

# Đánh Giá Random Forest
rf_model_eval = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_eval.fit(X_train.values, y_train)
rf_pred_eval = rf_model_eval.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred_eval)
rf_rmse = sqrt(mean_squared_error(y_test, rf_pred_eval))
print(f"Random Forest Evaluation - MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}")

# Retrain trên toàn bộ dữ liệu cho Dự Báo
model = LinearRegression()
model.fit(X.values, y)

# Dự Báo Cho 6 Tháng Tiếp Theo
last_month_index = monthly_data['month_index'].max()
future_indices = np.arange(last_month_index + 1, last_month_index + 7).reshape(-1, 1)
predictions = model.predict(future_indices)

# Lưu Kết Quả Linear Regression
last_date = monthly_data['sale_date'].max()
print("--- Lưu Dự Báo Linear Regression ---")

for i, pred in enumerate(predictions):
    forecast_date_obj = (last_date + timedelta(days=32*(i+1))).replace(day=1)
    forecast_date_str = forecast_date_obj.strftime('%Y-%m-%d')
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    predicted_val = float(pred) 
    predicted_val = max(0, predicted_val)

    sql_insert = """
    INSERT INTO Forecasts (product_id, forecast_date, predicted_quantity, model_name, created_at)
    VALUES (?, ?, ?, ?, ?)
    """
    
    cursor.execute(sql_insert, (1, forecast_date_str, predicted_val, 'Linear Regression', now_str))

# Retrain Random Forest trên toàn bộ
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X.values, y)
rf_predictions = rf_model.predict(future_indices)

print("--- Lưu Dự Báo Random Forest ---")

for i, pred in enumerate(rf_predictions):
    forecast_date_obj = (last_date + timedelta(days=32*(i+1))).replace(day=1)
    forecast_date_str = forecast_date_obj.strftime('%Y-%m-%d')
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    predicted_val = float(pred) 
    predicted_val = max(0, predicted_val)

    sql_insert = """
    INSERT INTO Forecasts (product_id, forecast_date, predicted_quantity, model_name, created_at)
    VALUES (?, ?, ?, ?, ?)
    """
    
    cursor.execute(sql_insert, (1, forecast_date_str, predicted_val, 'Random Forest', now_str))

conn.commit()
print("Thành công! Đã cập nhật bảng Forecasts với Linear Regression và Random Forest.")

--- Đang lưu kết quả dự báo ---
--- Dự báo Random Forest ---
Thành công! Đã cập nhật bảng Forecasts với cả Linear Regression và Random Forest.


In [ ]:
# Mô Hình LSTM
y = monthly_data['revenue']
last_date = monthly_data['sale_date'].max()

# Tiền Xử Lý: Chuẩn Hóa Dữ Liệu và Tạo Sequences
scaler = MinMaxScaler(feature_range=(0, 1))
y_scaled = scaler.fit_transform(y.values.reshape(-1, 1))

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])  # X = [t-3, t-2, t-1]
        y.append(data[i+seq_length])    # y = t
    return np.array(X), np.array(y)

seq_length = 3
X_seq, y_seq = create_sequences(y_scaled, seq_length)

# Chia Dữ Liệu Train/Test
if len(X_seq) > 5:
    train_size = int(len(X_seq) * 0.8)
    X_train, X_test = X_seq[:train_size], X_seq[train_size:]
    y_train, y_test = y_seq[:train_size], y_seq[train_size:]
else:
    X_train, y_train = X_seq, y_seq
    X_test, y_test = X_seq, y_seq

# Xây Dựng và Huấn Luyện Mô Hình LSTM
lstm_model = Sequential()
lstm_model.add(LSTM(50, activation='relu', input_shape=(seq_length, 1)))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X_train, y_train, epochs=50, verbose=0)

# Đánh Giá LSTM trên Test Set
if len(X_test) > 0:
    lstm_pred_scaled = lstm_model.predict(X_test, verbose=0)
    lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
    y_test_actual = scaler.inverse_transform(y_test).flatten()
    
    lstm_mae = mean_absolute_error(y_test_actual, lstm_pred)
    lstm_rmse = sqrt(mean_squared_error(y_test_actual, lstm_pred))
    print(f"LSTM Evaluation - MAE: {lstm_mae:.2f}, RMSE: {lstm_rmse:.2f}")
else:
    print("LSTM - Không đủ dữ liệu test để đánh giá")

# Retrain trên toàn bộ dữ liệu cho Dự Báo
X_seq_full, y_seq_full = create_sequences(y_scaled, seq_length)
lstm_model_full = Sequential()
lstm_model_full.add(LSTM(50, activation='relu', input_shape=(seq_length, 1)))
lstm_model_full.add(Dense(1))
lstm_model_full.compile(optimizer='adam', loss='mse')
lstm_model_full.fit(X_seq_full, y_seq_full, epochs=50, verbose=0)

# Dự Báo Cho 6 Tháng Tiếp Theo
last_sequence = y_scaled[-seq_length:].reshape(1, seq_length, 1)
lstm_predictions = []

for _ in range(6):
    lstm_pred = lstm_model_full.predict(last_sequence, verbose=0)
    lstm_predictions.append(lstm_pred[0][0])
    last_sequence = np.append(last_sequence[:, 1:, :], lstm_pred.reshape(1, 1, 1), axis=1)

# Đảo Ngược Chuẩn Hóa Dự Báo
lstm_predictions = scaler.inverse_transform(np.array(lstm_predictions).reshape(-1, 1)).flatten()

print("--- Lưu Dự Báo LSTM ---")
for i, pred in enumerate(lstm_predictions):
    forecast_date_obj = (last_date + timedelta(days=32*(i+1))).replace(day=1)
    forecast_date_str = forecast_date_obj.strftime('%Y-%m-%d')
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    predicted_val = float(pred)
    predicted_val = max(0, predicted_val)

    sql_insert = """
    INSERT INTO Forecasts (product_id, forecast_date, predicted_quantity, model_name, created_at)
    VALUES (?, ?, ?, ?, ?)
    """
    
    cursor.execute(sql_insert, (1, forecast_date_str, predicted_val, 'LSTM', now_str))

conn.commit()
print("Thành công! Đã cập nhật bảng Forecasts với LSTM.")

--- Dự báo LSTM ---
--- Dự báo ARIMA ---
Thành công! Đã cập nhật bảng Forecasts với LSTM và ARIMA.


In [ ]:
# --- Đánh Giá Mô Hình ARIMA ---
# 1. Huấn luyện ARIMA trên tập Train (Dùng chung y_train, y_test đã khai báo phía trước)
arima_model_eval = ARIMA(y_train.values, order=(1, 1, 1))
arima_model_fit = arima_model_eval.fit()
arima_pred_eval = arima_model_fit.forecast(steps=len(y_test))

# Tính toán sai số
arima_mae = mean_absolute_error(y_test, arima_pred_eval)
arima_rmse = sqrt(mean_squared_error(y_test, arima_pred_eval))
print(f"ARIMA Evaluation - MAE: {arima_mae:.2f}, RMSE: {arima_rmse:.2f}")

# 2. Retrain ARIMA trên toàn bộ dữ liệu (y) để Dự Báo 6 tháng tiếp theo
arima_model_full = ARIMA(y.values, order=(1, 1, 1))
arima_model_full_fit = arima_model_full.fit()
arima_predictions = arima_model_full_fit.forecast(steps=6)

print("--- Lưu Dự Báo ARIMA ---")
last_date = monthly_data['sale_date'].max()

for i, pred in enumerate(arima_predictions):
    # Tính ngày dự báo (mỗi tháng tiếp theo)
    forecast_date_obj = (last_date + timedelta(days=32*(i+1))).replace(day=1)
    forecast_date_str = forecast_date_obj.strftime('%Y-%m-%d')
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    predicted_val = float(pred)
    predicted_val = max(0, predicted_val) # Không cho phép dự báo số âm

    sql_insert = """
    INSERT INTO Forecasts (product_id, forecast_date, predicted_quantity, model_name, created_at)
    VALUES (?, ?, ?, ?, ?)
    """
    cursor.execute(sql_insert, (1, forecast_date_str, predicted_val, 'ARIMA', now_str))

# Xác nhận tác vụ và ngắt kết nối DB
conn.commit()
print("Thành công! Đã chốt lưu kết quả ARIMA vào Database và đóng Server.")
conn.close()
